# 2. Store Selection

Cross-tabulate `store.csv` on StoreType / Promo2 / CompetitionDistance and select a stratified set of 6 stores that covers meaningful variation on those axes, restricted to stores with (nearly) complete sales history. Corresponds to step 2 of the workflow in `CLAUDE.md`; this is the scoping decision that lets the paper analyze forecast performance by store characteristic instead of using the full 1,115-store dataset.

In [ ]:
import sys, json
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from data import load_store, load_train
from store_selection import cross_tab, select_stores

OUTPUTS = Path.cwd().parent / "outputs"
(OUTPUTS / "tables").mkdir(parents=True, exist_ok=True)

store = load_store()
train = load_train()

## Cross-tabulation: StoreType x Promo2 x CompetitionDistance tier

In [ ]:
cross_tab(store)

## Selecting 6 stores

`select_stores` first drops stores whose train.csv row count is below 95% of the full calendar span (incomplete history), then greedily picks stores to cover StoreType, Promo2, and CompetitionDistance-tier facets one at a time - each pick is the store that covers the most still-missing facet, with ties broken toward a brand-new (StoreType, Promo2, tier) combination. This guarantees every StoreType appears at least once (the primary axis) while still spending the remaining picks on Promo2/distance contrast.

In [ ]:
result = select_stores(store, train, n=6)
result.justification

In [ ]:
result.justification.to_csv(OUTPUTS / "tables" / "store_selection.csv", index=False)
with open(OUTPUTS / "selected_stores.json", "w") as f:
    json.dump(result.store_ids, f)
result.store_ids

## Justification (for the paper)

Each selected store fills a gap in StoreType, Promo2, or CompetitionDistance-tier coverage that the previously-selected stores left open (see the `Reason` column above) - the first four picks establish one store per StoreType, and the last two add Promo2/distance contrast that repeating a StoreType allows. All six passed the completeness filter, so none require special handling of missing weeks beyond the routine dropped closed days.